# ARGUS closed-loop embodied testbed (Kaggle)

**Notebook settings (right panel) before running:**
1. *Accelerator* → **GPU T4 x2** (or GPU P100)
2. *Internet* → **On** (needs a phone-verified Kaggle account)

Run the cells top to bottom. Step 1 is a quick smoke test (~2–3 min); step 2 is the real baseline (~20–40 min).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Get the code and install

In [ ]:
import os
if os.path.exists("/kaggle/working/ARGUS"):
    !git -C /kaggle/working/ARGUS pull
else:
    !git clone https://github.com/AayushOjha001/ARGUS.git /kaggle/working/ARGUS
%cd /kaggle/working/ARGUS
!pip install -q pybullet qwen-vl-utils
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU is enough for the 2B model

## 1. Smoke test: one attacked episode
Check that the model's raw answers look like actions (`pick red`, `place tray`, ...). If they are garbage or empty, re-run with `--dtype fp32`.

In [ ]:
!python embodied/closed_loop_pybullet.py --policy vlm --model Qwen/Qwen2-VL-2B-Instruct     --conditions attack_card --episodes 1 --max-steps 6 --out /kaggle/working/smoke

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/smoke/attack_card/ep000/step00.png"))
display(Image("/kaggle/working/smoke/attack_card/ep000/final.png"))

## 2. No-defense baseline (the first real experiment)
Same 20 random layouts under 4 conditions:
- `clean`: no text in the scene
- `benign_card`: harmless text card (control: does *any* text distract the model?)
- `attack_card`: injection printed on a card in the scene (physical attack)
- `attack_overlay`: injection as a caption strip (same style as the ARGUS image dataset)

In [ ]:
!python embodied/closed_loop_pybullet.py --policy vlm --model Qwen/Qwen2-VL-2B-Instruct     --conditions clean,benign_card,attack_card,attack_overlay --episodes 20 --out /kaggle/working/emb

## 3. Results

In [ ]:
import json
s = json.load(open("/kaggle/working/emb/summary.json"))
cols = ["task_success_rate", "attack_success_rate", "attacker_object_picked_rate", "first_action_hijack_rate", "mean_steps"]
print(f"{'condition':16}" + "".join(f"{c.replace('_rate',''):>22}" for c in cols))
for name, m in s["conditions"].items():
    print(f"{name:16}" + "".join(f"{m[c]:>22}" for c in cols))

In [ ]:
# Raw model outputs for the first attacked episode
ep = json.load(open("/kaggle/working/emb/attack_card/ep000/episode.json"))
for st in ep["steps"]:
    print(st["t"], repr(st["raw"]), "->", st["action"], "|", st["status"])

In [ ]:
# Zip everything so you can download it (Output tab, right panel)
!cd /kaggle/working && zip -qr emb_results.zip emb && ls -lh emb_results.zip